In [2]:
from dataclasses import dataclass,field,asdict
import json
from typing import List, Dict
import models
import torch


In [3]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
from utils.validation import get_validation_recalls
from models import helper
from utils import utils
import models

In [4]:
import warnings
warnings.filterwarnings("ignore")

# Тестирование

In [6]:
conf = utils.load_from_json('conf.json')
for test_name in conf.tests:
    for model_conf in conf.models:
        model_conf = utils.ModelItem(**model_conf)
        print(f'------- {model_conf.title} -------')
        model = models.MODELS_PULL[model_conf.model_name](**model_conf.model_conf)
        model.load_model_state_dict(model_conf.model_weight_path)
        device = torch.device(model_conf.device)
        print(device)
        model.set_model_device(device)
        for dataset_name in conf.datasets:
            utils.run_test(model,dataset_name,test_name)

------- MIX VPR -------
cuda:0


Calculating descritptors...:   0%|          | 0/42 [00:00<?, ?it/s]

Descriptor dimension 4096


+------------------------------------+
|        Performance on essex        |
+----------+-------+--------+--------+
|    K     |   1   |   5    |   10   |
+----------+-------+--------+--------+
| Recall@K | 88.10 | 100.00 | 100.00 |
+----------+-------+--------+--------+


Calculating descritptors...:   0%|          | 0/3036 [00:00<?, ?it/s]

Descriptor dimension 4096


+----------------------------------+
|     Performance on nordland      |
+----------+-------+-------+-------+
|    K     |   1   |   5   |   10  |
+----------+-------+-------+-------+
| Recall@K | 55.11 | 70.33 | 75.47 |
+----------+-------+-------+-------+
------- BOQ VPR -------


...

Loaded model from /media/sunveil/Data/header_detection/vpr-auto-tests/dnv2_NV_AB_wpca8192_last.ckpt Successfully!
cuda:0


Calculating descritptors...:   0%|          | 0/42 [00:00<?, ?it/s]

Descriptor dimension 8192


+------------------------------------+
|        Performance on essex        |
+----------+-------+--------+--------+
|    K     |   1   |   5    |   10   |
+----------+-------+--------+--------+
| Recall@K | 91.90 | 100.00 | 100.00 |
+----------+-------+--------+--------+


Calculating descritptors...:   0%|          | 0/3036 [00:00<?, ?it/s]

Descriptor dimension 8192


+----------------------------------+
|     Performance on nordland      |
+----------+-------+-------+-------+
|    K     |   1   |   5   |   10  |
+----------+-------+-------+-------+
| Recall@K | 74.28 | 87.64 | 90.98 |
+----------+-------+-------+-------+


# Наборы данных

## ESSEX3IN1

![image.png](essex.jpeg)

Набор состоит из 420 фотографий (query = 210; reference = 210), представляет собой фотографии помещений, улиц и природных сцен.
Особенность набора в том, что он включает фотографии мест, которые являются запутанными как для VPR, так и для человеческого распознавания. Он содержит запутанные и сложные динамические объекты, естественные сцены и малоинформативные кадры. Как показано в нашей статье [«Запоминающиеся карты: A Framework for Re-defining Places in Visual Place Recognition»](https://arxiv.org/abs/1811.03529), большинство современных методов VPR с трудом справляются с этими запутанными изображениями.

Набор данных разделен на 2 папки. Соответствие между кадрами применяется к кадрам запроса и опорным кадрам. Изображения 0-132 в каждой папке являются запутанными, а изображения 133-209 - хорошими кадрами.
## Nordland

![image.png](nordland.jpg)

Набор состоит из 30352 фотографий (query = 2760; reference = 27592), представляет собой фотографии, собранные с поездов на железных дорогах Норвегии в разные времена суток и года. [Подробнее на ...](https://nrkbeta.no/2013/01/15/nordlandsbanen-minute-by-minute-season-by-season/)

# Модели

### MixVPR
![Описание изображения](./mixvpr.png)

### Vlad-Buff VPR
![Описание изображения](./vladbuff.png)

### BOQ
![Описание изображения](./BOQ.jpg)

# Выводы


|    Model Name      |   Embedding dimension   |
|:-----------------------------|:---------:|
|            MIX              |  4096  |
|         VLADBUFF            |  8192  | 
|            BOQ              |  12288  |




$$
Recall@k = \frac{Кол-во\ релевантных\ элементов\ в\ top-k}{Общее\ кол-во\ релевантных\ документов}
$$

Метрика используется, когда важно найти как можно больше релевантных элементов, даже если это означает включение некоторых нерелевантных.

### ESSEX
|    Recall@K on ESSEX      |   K=1   |   K=5   |   K=10  |
|-----------------------------|---------|---------|---------|
|            MIX              |  88.10  | *100.00*  | *100.00*  |
|         VLADBUFF            |  **91.90**  | *100.00*  | *100.00*  |
|            BOQ              |  90.95  |  99.52  | *100.00*  |
### NORDLAND
|    Recall@K on NORDLAND      |   K=1   |   K=5   |   K=10  |
|-----------------------------|---------|---------|---------|
|            MIX              |  55.11 | 70.33 | 75.47  |
|         VLADBUFF            |  74.28 | 87.64 | 90.98  |
|            BOQ              |  **81.49** | **92.39** | **94.75**  |